# 06 — Política de concessão baseada em limite e impacto financeiro

Este notebook define a política de concessão de crédito como uma **política de limite por rating**, combinando:

- rating interno (derivado do `pd_score`)
- multiplicador de renda por rating
- teto máximo por rating (calibrado pela base histórica)
- capacidade de pagamento (valor presente da parcela máxima)
- fatores de ajuste: restritivos, atividade do cliente, tempo de relacionamento
- impacto financeiro em três cenários de apetite de risco

**Entrada principal:** `data/processed/base_politica_validacao_com_score.parquet`  
**Saída principal:** `data/processed/base_simulacao_cenarios_politica.parquet`

## 1. Contexto e correção metodológica

### O score de PD não é a política

O `pd_score` mede a **probabilidade estimada de inadimplência em 12 meses**. Ele é um insumo de risco, não uma decisão de crédito.

O fluxo correto é:

```
pd_score → faixa_risco (rating interno) → política de limite
```

A política deve definir:
- **qual público** pode receber crédito automaticamente;
- **qual limite** pode ser concedido;
- **qual impacto financeiro** a política gera;
- **qual risco** está sendo aceito por rating.

### Lógica de limite

Para cada cliente:

```
limite_multiplicador = renda × multiplicador_rating
limite_capacidade    = VP(parcela_maxima, taxa, prazo)
limite_bruto         = min(limite_multiplicador, limite_capacidade, teto_rating)
limite_final         = limite_bruto × fator_restritivo × fator_ativo × fator_tempo
```

### Decisões

- **Aprovar valor solicitado**: rating elegível + `valor_emprestado <= limite_final`
- **Aprovar valor reduzido**: rating elegível + `limite_final >= valor_minimo` + `valor_emprestado > limite_final`
- **Análise manual**: rating D ou condição de mesa
- **Recusar**: rating E, renda inválida ou limite abaixo do mínimo operacional

### O `target_inadimplente_12m` é usado apenas para backtest — nunca como regra de decisão.

## 2. Setup e leitura da base com score

In [1]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

PROJ_ROOT = Path('..').resolve()
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))

from src.concessao_credito.config import PROCESSED_DIR, TABLES_DIR, FIGURES_DIR
from src.concessao_credito.plots import (
    registrar_template_credito,
    aplicar_layout,
    salvar_figura,
    CORES,
)

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

registrar_template_credito()

print(f'TABLES_DIR  : {TABLES_DIR}')
print(f'FIGURES_DIR : {FIGURES_DIR}')

TABLES_DIR  : C:\GitHub\datascience\projetos\concessao_credito\outputs\tables
FIGURES_DIR : C:\GitHub\datascience\projetos\concessao_credito\outputs\figures


In [2]:
ARQUIVO_BASE = PROCESSED_DIR / 'base_politica_validacao_com_score.parquet'

if not ARQUIVO_BASE.exists():
    raise FileNotFoundError(
        f'Base principal nao encontrada: {ARQUIVO_BASE}\n'
        'Execute o notebook 05 antes de prosseguir.'
    )

df = pd.read_parquet(ARQUIVO_BASE)

RATINGS_ORDER = [
    'A - Baixo risco',
    'B - Médio-baixo risco',
    'C - Médio risco',
    'D - Alto risco',
    'E - Muito alto risco',
]

df['faixa_risco'] = pd.Categorical(
    df['faixa_risco'], categories=RATINGS_ORDER, ordered=True
)

print(f'Base carregada: {df.shape[0]:,} linhas x {df.shape[1]} colunas')
print()
print('Distribuicao por rating:')
print(df['faixa_risco'].value_counts().sort_index())

Base carregada: 4,862 linhas x 41 colunas

Distribuicao por rating:
faixa_risco
A - Baixo risco          1945
B - Médio-baixo risco     972
C - Médio risco           729
D - Alto risco            729
E - Muito alto risco      487
Name: count, dtype: int64


## 3. Contrato de dados e validações iniciais

In [3]:
COLUNAS_OBRIGATORIAS = [
    'id_cliente', 'pd_score', 'faixa_risco',
    'valor_renda', 'valor_emprestado', 'valor_parcela',
    'valor_taxa', 'valor_prazo', 'valor_restritivos',
    'restritivos_sobre_renda', 'comprometimento_renda',
    'tempo_conta_anos', 'flag_cliente_ativo', 'target_inadimplente_12m',
]

faltando = [c for c in COLUNAS_OBRIGATORIAS if c not in df.columns]
if faltando:
    raise ValueError(f'Colunas obrigatorias ausentes: {faltando}')
print('OK — todas as colunas obrigatorias presentes')

cols_numericas = [
    'pd_score', 'valor_renda', 'valor_emprestado', 'valor_parcela',
    'valor_taxa', 'valor_prazo', 'valor_restritivos',
    'restritivos_sobre_renda', 'comprometimento_renda',
    'tempo_conta_anos', 'flag_cliente_ativo', 'target_inadimplente_12m',
]
for col in cols_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# pd_score must be in [0, 1]
score_min = df['pd_score'].min()
score_max = df['pd_score'].max()
if not df['pd_score'].between(0, 1).all():
    raise ValueError(f'pd_score fora de [0,1]: min={score_min:.4f}, max={score_max:.4f}')
print(f'OK — pd_score em [{score_min:.4f}, {score_max:.4f}]')

# faixa_risco categories
faixas_encontradas = set(df['faixa_risco'].cat.categories)
faixas_esperadas = set(RATINGS_ORDER)
if not faixas_encontradas.issubset(faixas_esperadas):
    print(f'Atencao — categorias inesperadas em faixa_risco: {faixas_encontradas - faixas_esperadas}')
print(f'OK — faixa_risco: {df["faixa_risco"].nunique()} categorias validas')

# Null check
nulos = df[cols_numericas].isnull().sum()
nulos_com_problema = nulos[nulos > 0]
if len(nulos_com_problema) > 0:
    print('Atencao — nulos nas colunas numericas:')
    print(nulos_com_problema)
else:
    print('OK — sem nulos nas colunas numericas obrigatorias')

# Create id_operacao if missing
if 'id_operacao' not in df.columns:
    df['id_operacao'] = df.index.astype(str).str.zfill(6)
    print('id_operacao criado a partir do indice (cada cliente tem uma operacao na base)')

print()
print(f'Base validada: {df.shape[0]:,} registros prontos para simulacao')

OK — todas as colunas obrigatorias presentes
OK — pd_score em [0.0047, 0.9214]
OK — faixa_risco: 5 categorias validas
OK — sem nulos nas colunas numericas obrigatorias

Base validada: 4,862 registros prontos para simulacao


In [4]:
# --- Variáveis auxiliares ---

def classificar_faixa_renda(valor):
    if pd.isna(valor) or valor <= 0:
        return 'Sem renda'
    elif valor <= 2000:
        return 'Ate R$ 2 mil'
    elif valor <= 5000:
        return 'R$ 2 mil a R$ 5 mil'
    elif valor <= 10000:
        return 'R$ 5 mil a R$ 10 mil'
    elif valor <= 20000:
        return 'R$ 10 mil a R$ 20 mil'
    else:
        return 'Acima de R$ 20 mil'

ORDEM_FAIXA_RENDA = [
    'Sem renda',
    'Ate R$ 2 mil',
    'R$ 2 mil a R$ 5 mil',
    'R$ 5 mil a R$ 10 mil',
    'R$ 10 mil a R$ 20 mil',
    'Acima de R$ 20 mil',
]

def classificar_tempo_relacionamento(anos):
    if pd.isna(anos) or anos < 1:
        return 'curto'
    elif anos < 3:
        return 'medio'
    else:
        return 'longo'

def classificar_restritivo_sobre_renda(pct):
    if pd.isna(pct) or pct <= 0:
        return 'sem_restritivo'
    elif pct <= 0.02:
        return 'ate_2pct'
    elif pct <= 0.05:
        return 'ate_5pct'
    elif pct <= 0.10:
        return 'ate_10pct'
    else:
        return 'acima_10pct'

ORDEM_RESTRITIVO = ['sem_restritivo', 'ate_2pct', 'ate_5pct', 'ate_10pct', 'acima_10pct']
ORDEM_TEMPO_REL  = ['curto', 'medio', 'longo']

df['faixa_renda'] = pd.Categorical(
    df['valor_renda'].apply(classificar_faixa_renda),
    categories=ORDEM_FAIXA_RENDA, ordered=True
)
df['classe_restritivo'] = pd.Categorical(
    df['restritivos_sobre_renda'].apply(classificar_restritivo_sobre_renda),
    categories=ORDEM_RESTRITIVO, ordered=True
)
df['classe_tempo_relacionamento'] = pd.Categorical(
    df['tempo_conta_anos'].apply(classificar_tempo_relacionamento),
    categories=ORDEM_TEMPO_REL, ordered=True
)

print('Variaveis auxiliares criadas:')
print('  faixa_renda:')
print(df['faixa_renda'].value_counts().sort_index())
print()
print('  classe_restritivo:')
print(df['classe_restritivo'].value_counts().sort_index())
print()
print('  classe_tempo_relacionamento:')
print(df['classe_tempo_relacionamento'].value_counts().sort_index())

Variaveis auxiliares criadas:
  faixa_renda:
faixa_renda
Sem renda                   0
Ate R$ 2 mil             1173
R$ 2 mil a R$ 5 mil      2721
R$ 5 mil a R$ 10 mil      829
R$ 10 mil a R$ 20 mil     135
Acima de R$ 20 mil          4
Name: count, dtype: int64

  classe_restritivo:
classe_restritivo
sem_restritivo    1795
ate_2pct          1509
ate_5pct           873
ate_10pct          470
acima_10pct        215
Name: count, dtype: int64

  classe_tempo_relacionamento:
classe_tempo_relacionamento
curto     730
medio    3568
longo     564
Name: count, dtype: int64


## 4. Caracterização do público da política

Antes de propor a política, é necessário entender quem é o público: distribuição de renda, risco, restritivos, exposição e tempo de relacionamento.

In [5]:
# 4.1 Faixa de renda × Rating
tab_renda_rating = (
    df.groupby(['faixa_renda', 'faixa_risco'], observed=True)
    .size()
    .reset_index(name='qtd_clientes')
)
tab_renda_rating.to_csv(
    TABLES_DIR / 'politica_publico_faixa_renda_rating.csv', index=False
)

pivot_renda = tab_renda_rating.pivot(
    index='faixa_renda', columns='faixa_risco', values='qtd_clientes'
).fillna(0).astype(int)

print('Tabela 4.1 — Faixa de renda x Rating (qtd clientes):')
print(pivot_renda)
print()
print('Salva em: politica_publico_faixa_renda_rating.csv')

Tabela 4.1 — Faixa de renda x Rating (qtd clientes):
faixa_risco            A - Baixo risco  B - Médio-baixo risco  \
faixa_renda                                                     
Ate R$ 2 mil                       458                    222   
R$ 2 mil a R$ 5 mil               1070                    559   
R$ 5 mil a R$ 10 mil               353                    154   
R$ 10 mil a R$ 20 mil               62                     35   
Acima de R$ 20 mil                   2                      2   

faixa_risco            C - Médio risco  D - Alto risco  E - Muito alto risco  
faixa_renda                                                                   
Ate R$ 2 mil                       173             206                   114  
R$ 2 mil a R$ 5 mil                419             384                   289  
R$ 5 mil a R$ 10 mil               121             120                    81  
R$ 10 mil a R$ 20 mil               16              19                     3  
Acima de R$ 20 mi

In [6]:
# 4.2 Rating × Tempo de relacionamento
tab_rating_tempo = (
    df.groupby(['faixa_risco', 'classe_tempo_relacionamento'], observed=True)
    .size()
    .reset_index(name='qtd_clientes')
)
tab_rating_tempo.to_csv(
    TABLES_DIR / 'politica_publico_rating_tempo_relacionamento.csv', index=False
)

pivot_tempo = tab_rating_tempo.pivot(
    index='faixa_risco', columns='classe_tempo_relacionamento', values='qtd_clientes'
).fillna(0).astype(int)

print('Tabela 4.2 — Rating x Tempo de relacionamento (qtd clientes):')
print(pivot_tempo)
print()
print('Salva em: politica_publico_rating_tempo_relacionamento.csv')

Tabela 4.2 — Rating x Tempo de relacionamento (qtd clientes):
classe_tempo_relacionamento  curto  medio  longo
faixa_risco                                     
A - Baixo risco                156   1520    269
B - Médio-baixo risco          166    663    143
C - Médio risco                104    562     63
D - Alto risco                 170    498     61
E - Muito alto risco           134    325     28

Salva em: politica_publico_rating_tempo_relacionamento.csv


In [7]:
# 4.3 Rating × PD e bad rate
tab_rating_pd = (
    df.groupby('faixa_risco', observed=True)
    .agg(
        qtd_clientes=('pd_score', 'count'),
        pd_min=('pd_score', 'min'),
        pd_media=('pd_score', 'mean'),
        pd_max=('pd_score', 'max'),
        bad_rate_observado=('target_inadimplente_12m', 'mean'),
    )
    .reset_index()
)
tab_rating_pd.to_csv(
    TABLES_DIR / 'politica_publico_rating_pd_bad_rate.csv', index=False
)

print('Tabela 4.3 — Rating x PD e bad rate observado:')
print(tab_rating_pd.round(4).to_string(index=False))
print()
print('Salva em: politica_publico_rating_pd_bad_rate.csv')

Tabela 4.3 — Rating x PD e bad rate observado:
          faixa_risco  qtd_clientes  pd_min  pd_media  pd_max  bad_rate_observado
      A - Baixo risco          1945  0.0047    0.0225  0.0425              0.0108
B - Médio-baixo risco           972  0.0426    0.0607  0.0823              0.0453
      C - Médio risco           729  0.0823    0.1098  0.1465              0.1056
       D - Alto risco           729  0.1467    0.2253  0.3504              0.2510
 E - Muito alto risco           487  0.3505    0.5224  0.9214              0.5606

Salva em: politica_publico_rating_pd_bad_rate.csv


In [8]:
# 4.4 Rating × Exposição e capacidade
tab_rating_exp = (
    df.groupby('faixa_risco', observed=True)
    .agg(
        qtd_clientes=('valor_emprestado', 'count'),
        renda_media=('valor_renda', 'mean'),
        valor_solicitado_medio=('valor_emprestado', 'mean'),
        valor_solicitado_total=('valor_emprestado', 'sum'),
        parcela_media=('valor_parcela', 'mean'),
        comprometimento_medio=('comprometimento_renda', 'mean'),
        restritivos_media=('valor_restritivos', 'mean'),
        restritivos_sobre_renda_media=('restritivos_sobre_renda', 'mean'),
    )
    .reset_index()
)
tab_rating_exp.to_csv(
    TABLES_DIR / 'politica_publico_rating_exposicao.csv', index=False
)

print('Tabela 4.4 — Rating x Exposicao e capacidade:')
print(tab_rating_exp.round(2).to_string(index=False))
print()
print('Salva em: politica_publico_rating_exposicao.csv')

Tabela 4.4 — Rating x Exposicao e capacidade:
          faixa_risco  qtd_clientes  renda_media  valor_solicitado_medio  valor_solicitado_total  parcela_media  comprometimento_medio  restritivos_media  restritivos_sobre_renda_media
      A - Baixo risco          1945      3761.76                10940.39             21279049.32        1508.15                   0.39              25.71                           0.01
B - Médio-baixo risco           972      3759.24                14628.84             14219232.37        1997.98                   0.52              59.49                           0.02
      C - Médio risco           729      3605.27                14963.62             10908478.17        2024.96                   0.56              76.38                           0.03
       D - Alto risco           729      3494.35                14768.03             10765895.77        2106.45                   0.60              85.37                           0.03
 E - Muito alto risco        

In [9]:
# 4.5 Rating × Restritivos
tab_rating_rest = (
    df.groupby(['faixa_risco', 'classe_restritivo'], observed=True)
    .size()
    .reset_index(name='qtd_clientes')
)
tab_rating_rest.to_csv(
    TABLES_DIR / 'politica_publico_rating_restritivos.csv', index=False
)

pivot_rest = tab_rating_rest.pivot(
    index='faixa_risco', columns='classe_restritivo', values='qtd_clientes'
).fillna(0).astype(int)

print('Tabela 4.5 — Rating x Classe de restritivo (qtd clientes):')
print(pivot_rest)
print()
print('Salva em: politica_publico_rating_restritivos.csv')

Tabela 4.5 — Rating x Classe de restritivo (qtd clientes):
classe_restritivo      sem_restritivo  ate_2pct  ate_5pct  ate_10pct  \
faixa_risco                                                            
A - Baixo risco                   981       690       185         65   
B - Médio-baixo risco             343       319       172         98   
C - Médio risco                   235       205       153         89   
D - Alto risco                    175       203       189        112   
E - Muito alto risco               61        92       174        106   

classe_restritivo      acima_10pct  
faixa_risco                         
A - Baixo risco                 24  
B - Médio-baixo risco           40  
C - Médio risco                 47  
D - Alto risco                  50  
E - Muito alto risco            54  

Salva em: politica_publico_rating_restritivos.csv


## 5. Validação do rating interno

O rating deve apresentar **monotonicidade**: PD médio e bad rate crescentes de A para E. Isso valida que o score ordena risco de forma correta.

In [10]:
df_val_rating = (
    df.groupby('faixa_risco', observed=True)
    .agg(
        qtd_clientes=('pd_score', 'count'),
        pd_min=('pd_score', 'min'),
        pd_media=('pd_score', 'mean'),
        pd_max=('pd_score', 'max'),
        bad_rate_observado=('target_inadimplente_12m', 'mean'),
        valor_solicitado_total=('valor_emprestado', 'sum'),
    )
    .reset_index()
)
df_val_rating['pct_clientes'] = (
    df_val_rating['qtd_clientes'] / df_val_rating['qtd_clientes'].sum()
)

print('Rating interno — estatisticas de validacao:')
print(df_val_rating.round(4).to_string(index=False))
print()

pd_medias  = df_val_rating['pd_media'].values
bad_rates  = df_val_rating['bad_rate_observado'].values
mono_pd = all(pd_medias[i] <= pd_medias[i + 1] for i in range(len(pd_medias) - 1))
mono_br = all(bad_rates[i] <= bad_rates[i + 1] for i in range(len(bad_rates) - 1))

print(f'Monotonicidade pd_media  A->E : {"OK" if mono_pd else "VERIFICAR"}')
print(f'Monotonicidade bad_rate  A->E : {"OK" if mono_br else "VERIFICAR"}')

Rating interno — estatisticas de validacao:
          faixa_risco  qtd_clientes  pd_min  pd_media  pd_max  bad_rate_observado  valor_solicitado_total  pct_clientes
      A - Baixo risco          1945  0.0047    0.0225  0.0425              0.0108             21279049.32        0.4000
B - Médio-baixo risco           972  0.0426    0.0607  0.0823              0.0453             14219232.37        0.1999
      C - Médio risco           729  0.0823    0.1098  0.1465              0.1056             10908478.17        0.1499
       D - Alto risco           729  0.1467    0.2253  0.3504              0.2510             10765895.77        0.1499
 E - Muito alto risco           487  0.3505    0.5224  0.9214              0.5606              7431711.59        0.1002

Monotonicidade pd_media  A->E : OK
Monotonicidade bad_rate  A->E : OK


## 6. Definição dos parâmetros de limite

### Estrutura da política de limite

Cada cenário define por rating:

| Parâmetro | Descrição |
|---|---|
| `multiplicador_renda` | Fator da renda para o limite máximo por renda |
| `teto_rating` | Valor máximo absoluto por rating (calibrado por percentis históricos) |
| `pct_max_comprometimento` | Percentual máximo da renda que pode ser comprometido com a parcela |
| `tratamento_rating` | `automatico`, `analise_manual` ou `recusa` |

Fatores de ajuste (≤ 1,0):

| Fator | Descrição |
|---|---|
| `fator_restritivo` | Reduz o limite conforme a classe de restritivos |
| `fator_cliente_ativo` | Reduz o limite para clientes inativos |
| `fator_tempo_relacionamento` | Reduz o limite para clientes com pouco tempo de conta |

### Calibração dos tetos

Os tetos são calculados a partir de **percentis históricos do `valor_emprestado` por rating**:
- Conservador: percentil 60
- Base/Equilibrado: percentil 75
- Expansivo controlado: percentil 90

In [11]:
# Calibracao dos tetos por percentis historicos do valor_emprestado por rating.
# A calibracao usa a base historica como ancora, mas impoe monotonicidade de politica:
# A > B > C > D/E. D e E seguem sem limite automatico.
PERCENTIL_TETO = {
    'Conservador':          0.60,
    'Base/Equilibrado':     0.75,
    'Expansivo controlado': 0.90,
}

FATOR_TETO_ELEGIVEL = {
    'A - Baixo risco':       1.25,
    'B - Médio-baixo risco': 1.00,
    'C - Médio risco':       0.75,
}


def calibrar_tetos_monotonicos(tetos_observados):
    """Calibra tetos finitos e monotonicos para ratings elegiveis."""
    ratings_elegiveis = ['A - Baixo risco', 'B - Médio-baixo risco', 'C - Médio risco']
    teto_referencia = max(float(tetos_observados.get(r, 0.0) or 0.0) for r in ratings_elegiveis)

    tetos = {
        rating: round(teto_referencia * fator, 2)
        for rating, fator in FATOR_TETO_ELEGIVEL.items()
    }
    tetos['D - Alto risco'] = 0.0
    tetos['E - Muito alto risco'] = 0.0
    return tetos


tetos_por_cenario = {}
for cenario_nome, pct in PERCENTIL_TETO.items():
    tetos_observados = (
        df.groupby('faixa_risco', observed=True)['valor_emprestado']
        .quantile(pct)
        .to_dict()
    )
    tetos_por_cenario[cenario_nome] = calibrar_tetos_monotonicos(tetos_observados)

print('Tetos calibrados por cenario (R$):')
print()
for cenario_nome, teto in tetos_por_cenario.items():
    pct = PERCENTIL_TETO[cenario_nome]
    print(f'{cenario_nome} (p{int(pct*100)} + monotonicidade A>B>C):')
    for r in RATINGS_ORDER:
        v = teto.get(r, 0)
        print(f'  {r}: R$ {v:,.0f}')
    print()

Tetos calibrados por cenario (R$):



Conservador (p60 + monotonicidade A>B>C):
  A - Baixo risco: R$ 17,987
  B - Médio-baixo risco: R$ 14,389
  C - Médio risco: R$ 10,792
  D - Alto risco: R$ 0
  E - Muito alto risco: R$ 0

Base/Equilibrado (p75 + monotonicidade A>B>C):
  A - Baixo risco: R$ 25,365
  B - Médio-baixo risco: R$ 20,292
  C - Médio risco: R$ 15,219
  D - Alto risco: R$ 0
  E - Muito alto risco: R$ 0

Expansivo controlado (p90 + monotonicidade A>B>C):
  A - Baixo risco: R$ 39,223
  B - Médio-baixo risco: R$ 31,378
  C - Médio risco: R$ 23,534
  D - Alto risco: R$ 0
  E - Muito alto risco: R$ 0



In [12]:
CENARIOS = {
    'Conservador': {
        'multiplicadores': {
            'A - Baixo risco':       3.0,
            'B - Médio-baixo risco': 2.0,
            'C - Médio risco':       1.5,
            'D - Alto risco':        0.0,
            'E - Muito alto risco':  0.0,
        },
        'pct_max_comprometimento': {
            'A - Baixo risco':       0.25,
            'B - Médio-baixo risco': 0.20,
            'C - Médio risco':       0.15,
            'D - Alto risco':        0.10,
            'E - Muito alto risco':  0.00,
        },
        'teto_rating': tetos_por_cenario['Conservador'],
        'tratamento_rating': {
            'A - Baixo risco':       'automatico',
            'B - Médio-baixo risco': 'automatico',
            'C - Médio risco':       'automatico',
            'D - Alto risco':        'analise_manual',
            'E - Muito alto risco':  'recusa',
        },
        'fatores_restritivo': {
            'sem_restritivo': 1.00,
            'ate_2pct':       0.90,
            'ate_5pct':       0.75,
            'ate_10pct':      0.55,
            'acima_10pct':    0.00,
        },
        'fatores_cliente_ativo': {1: 1.00, 0: 0.70},
        'fatores_tempo_relacionamento': {'longo': 1.00, 'medio': 0.85, 'curto': 0.75},
        'valor_minimo_operacional': 500.0,
    },
    'Base/Equilibrado': {
        'multiplicadores': {
            'A - Baixo risco':       4.0,
            'B - Médio-baixo risco': 3.0,
            'C - Médio risco':       2.0,
            'D - Alto risco':        0.0,
            'E - Muito alto risco':  0.0,
        },
        'pct_max_comprometimento': {
            'A - Baixo risco':       0.30,
            'B - Médio-baixo risco': 0.25,
            'C - Médio risco':       0.20,
            'D - Alto risco':        0.15,
            'E - Muito alto risco':  0.00,
        },
        'teto_rating': tetos_por_cenario['Base/Equilibrado'],
        'tratamento_rating': {
            'A - Baixo risco':       'automatico',
            'B - Médio-baixo risco': 'automatico',
            'C - Médio risco':       'automatico',
            'D - Alto risco':        'analise_manual',
            'E - Muito alto risco':  'recusa',
        },
        'fatores_restritivo': {
            'sem_restritivo': 1.00,
            'ate_2pct':       1.00,
            'ate_5pct':       0.90,
            'ate_10pct':      0.70,
            'acima_10pct':    0.50,
        },
        'fatores_cliente_ativo': {1: 1.00, 0: 0.80},
        'fatores_tempo_relacionamento': {'longo': 1.00, 'medio': 0.90, 'curto': 0.80},
        'valor_minimo_operacional': 500.0,
    },
    'Expansivo controlado': {
        'multiplicadores': {
            'A - Baixo risco':       5.0,
            'B - Médio-baixo risco': 4.0,
            'C - Médio risco':       2.5,
            'D - Alto risco':        0.0,
            'E - Muito alto risco':  0.0,
        },
        'pct_max_comprometimento': {
            'A - Baixo risco':       0.35,
            'B - Médio-baixo risco': 0.30,
            'C - Médio risco':       0.25,
            'D - Alto risco':        0.20,
            'E - Muito alto risco':  0.00,
        },
        'teto_rating': tetos_por_cenario['Expansivo controlado'],
        'tratamento_rating': {
            'A - Baixo risco':       'automatico',
            'B - Médio-baixo risco': 'automatico',
            'C - Médio risco':       'automatico',
            'D - Alto risco':        'analise_manual',
            'E - Muito alto risco':  'recusa',
        },
        'fatores_restritivo': {
            'sem_restritivo': 1.00,
            'ate_2pct':       1.00,
            'ate_5pct':       0.95,
            'ate_10pct':      0.80,
            'acima_10pct':    0.60,
        },
        'fatores_cliente_ativo': {1: 1.00, 0: 0.85},
        'fatores_tempo_relacionamento': {'longo': 1.00, 'medio': 0.95, 'curto': 0.85},
        'valor_minimo_operacional': 500.0,
    },
}

print('Cenarios definidos:', list(CENARIOS.keys()))

Cenarios definidos: ['Conservador', 'Base/Equilibrado', 'Expansivo controlado']


In [13]:
# Salvar tabela de parâmetros
rows_params = []
for cenario_nome, params in CENARIOS.items():
    for rating in RATINGS_ORDER:
        rows_params.append({
            'cenario':                   cenario_nome,
            'faixa_risco':               rating,
            'multiplicador_renda':       params['multiplicadores'].get(rating, 0.0),
            'teto_rating':               params['teto_rating'].get(rating, 0.0),
            'pct_max_comprometimento':   params['pct_max_comprometimento'].get(rating, 0.0),
            'elegivel_aprovacao_automatica': params['tratamento_rating'].get(rating, '') == 'automatico',
            'elegivel_aprovacao_reduzida':   params['tratamento_rating'].get(rating, '') == 'automatico',
            'tratamento_rating':         params['tratamento_rating'].get(rating, 'recusa'),
            'valor_minimo_operacional':  params['valor_minimo_operacional'],
        })

df_params = pd.DataFrame(rows_params)
df_params.to_csv(TABLES_DIR / 'politica_parametros_limite_cenarios.csv', index=False)

print('Parametros de limite por cenario e rating:')
print(df_params.to_string(index=False))
print()
print('Salvo em: politica_parametros_limite_cenarios.csv')

Parametros de limite por cenario e rating:
             cenario           faixa_risco  multiplicador_renda  teto_rating  pct_max_comprometimento  elegivel_aprovacao_automatica  elegivel_aprovacao_reduzida tratamento_rating  valor_minimo_operacional
         Conservador       A - Baixo risco                  3.0     17986.62                     0.25                           True                         True        automatico                     500.0
         Conservador B - Médio-baixo risco                  2.0     14389.29                     0.20                           True                         True        automatico                     500.0
         Conservador       C - Médio risco                  1.5     10791.97                     0.15                           True                         True        automatico                     500.0
         Conservador        D - Alto risco                  0.0         0.00                     0.10                          False 

## 7. Funções da política de limite

In [14]:
def calcular_valor_presente_parcelas(parcela_maxima, taxa, prazo):
    """Valor presente de uma anuidade: VP = PMT * ((1-(1+i)^-n)/i)."""
    if parcela_maxima <= 0:
        return 0.0
    taxa  = max(float(taxa),  0.0001)
    prazo = max(int(prazo),   1)
    vp = parcela_maxima * (1 - (1 + taxa) ** (-prazo)) / taxa
    return max(float(vp), 0.0)


def calcular_limite_cliente(row, params):
    """Retorna (limite_mult, limite_cap, limite_bruto, limite_final) para um cliente."""
    rating     = row['faixa_risco']
    renda      = row['valor_renda']
    taxa       = row['valor_taxa']
    prazo      = row['valor_prazo']
    classe_rest = row['classe_restritivo_cenario']
    classe_tempo = row['classe_tempo_relacionamento']
    ativo      = row['flag_cliente_ativo']

    if pd.isna(renda) or renda <= 0:
        return 0.0, 0.0, 0.0, 0.0

    mult     = params['multiplicadores'].get(rating, 0.0)
    pct_comp = params['pct_max_comprometimento'].get(rating, 0.0)
    teto     = params['teto_rating'].get(rating, 0.0)

    # 1. Limite por multiplicador de renda
    limite_mult = renda * mult

    # 2. Limite por capacidade de pagamento
    parcela_maxima = renda * pct_comp
    taxa_safe  = max(float(taxa)  if not pd.isna(taxa)  else 0.0001, 0.0001)
    prazo_safe = max(int(prazo)   if not pd.isna(prazo) else 1,      1)
    limite_cap = calcular_valor_presente_parcelas(parcela_maxima, taxa_safe, prazo_safe)

    # 3. Limite bruto = min(multiplicador, capacidade, teto)
    limite_bruto = min(limite_mult, limite_cap, teto)

    # 4. Fatores de ajuste
    f_rest  = params['fatores_restritivo'].get(str(classe_rest), 1.0)
    f_ativo = params['fatores_cliente_ativo'].get(int(ativo) if not pd.isna(ativo) else 0, 1.0)
    f_tempo = params['fatores_tempo_relacionamento'].get(str(classe_tempo), 1.0)

    # 5. Limite final
    limite_final = max(limite_bruto * f_rest * f_ativo * f_tempo, 0.0)

    return limite_mult, limite_cap, limite_bruto, limite_final


def aplicar_decisao(row, params):
    """Retorna (decisao, valor_aprovado) para um cliente num cenario."""
    rating       = row['faixa_risco']
    renda        = row['valor_renda']
    valor_sol    = row['valor_emprestado']
    limite_final = row['limite_final_cenario']
    valor_min    = params['valor_minimo_operacional']
    tratamento   = params['tratamento_rating'].get(rating, 'recusa')
    classe_rest  = str(row.get('classe_restritivo_cenario', 'sem_restritivo'))
    classe_tempo = str(row.get('classe_tempo_relacionamento', ''))
    ativo        = row.get('flag_cliente_ativo', 0)

    if tratamento == 'recusa':
        return 'Recusar', 0.0

    if tratamento == 'analise_manual':
        return 'Análise manual', 0.0

    if pd.isna(renda) or renda <= 0:
        return 'Recusar', 0.0

    if limite_final < valor_min:
        return 'Recusar', 0.0

    if rating == 'C - Médio risco':
        restritivo_relevante = classe_rest in ['ate_10pct', 'acima_10pct']
        cliente_inativo = pd.isna(ativo) or int(ativo) != 1
        relacionamento_curto = classe_tempo == 'curto'
        proposta_muito_acima_limite = valor_sol > limite_final * 1.50

        if restritivo_relevante or cliente_inativo or relacionamento_curto:
            return 'Análise manual', 0.0

        if valor_sol > limite_final and proposta_muito_acima_limite:
            return 'Análise manual', 0.0

    if valor_sol <= limite_final:
        return 'Aprovar valor solicitado', float(valor_sol)
    else:
        return 'Aprovar valor reduzido', float(limite_final)


def simular_politica_limite(df_entrada, cenario_nome, params):
    """Simula a politica de limite para um cenario inteiro."""
    res = df_entrada.copy()
    res['cenario'] = cenario_nome

    # Colunas especificas do cenario
    res['classe_restritivo_cenario']       = res['classe_restritivo'].astype(str)
    res['multiplicador_renda_cenario']     = res['faixa_risco'].map(params['multiplicadores']).fillna(0.0)
    res['teto_rating_cenario']             = res['faixa_risco'].map(params['teto_rating']).fillna(0.0)
    res['pct_max_comprometimento_cenario'] = res['faixa_risco'].map(params['pct_max_comprometimento']).fillna(0.0)

    # Calcular limites
    limites = res.apply(
        lambda row: calcular_limite_cliente(row, params), axis=1, result_type='expand'
    )
    res['limite_multiplicador_cenario'] = limites[0]
    res['limite_capacidade_cenario']    = limites[1]
    res['limite_bruto_cenario']         = limites[2]
    res['limite_final_cenario']         = limites[3]

    # Aplicar decisao
    decisoes = res.apply(
        lambda row: aplicar_decisao(row, params), axis=1, result_type='expand'
    )
    res['decisao_cenario']       = decisoes[0]
    res['valor_aprovado_cenario'] = decisoes[1]

    # Seguranca: valor aprovado nunca maior que solicitado
    mask_aprov = res['decisao_cenario'].isin(['Aprovar valor solicitado', 'Aprovar valor reduzido'])
    res.loc[mask_aprov, 'valor_aprovado_cenario'] = np.minimum(
        res.loc[mask_aprov, 'valor_aprovado_cenario'],
        res.loc[mask_aprov, 'valor_emprestado'],
    )

    return res


def validar_consistencia_politica(df_sim, cenario_nome):
    """Valida regras de consistencia obrigatorias."""
    erros = []

    mask_e = df_sim['faixa_risco'] == 'E - Muito alto risco'
    if not (df_sim.loc[mask_e, 'decisao_cenario'] == 'Recusar').all():
        erros.append('Rating E com decisao diferente de Recusar')

    mask_aprov = df_sim['decisao_cenario'].isin(['Aprovar valor solicitado', 'Aprovar valor reduzido'])
    if (df_sim.loc[mask_aprov, 'valor_aprovado_cenario'] > df_sim.loc[mask_aprov, 'valor_emprestado'] + 0.01).any():
        erros.append('Valor aprovado maior que valor solicitado')

    mask_zero = df_sim['decisao_cenario'].isin(['Análise manual', 'Recusar'])
    if (df_sim.loc[mask_zero, 'valor_aprovado_cenario'] > 0).any():
        erros.append('Analise manual ou Recusa com valor aprovado > 0')

    status = 'OK — consistencia validada' if not erros else f'ERROS: {erros}'
    print(f'  [{cenario_nome}] {status}')
    return len(erros) == 0


print('Funcoes da politica de limite definidas.')

Funcoes da politica de limite definidas.


## 8. Simulação dos cenários

In [15]:
COLS_SAIDA = [
    'cenario', 'id_cliente', 'id_operacao', 'pd_score', 'faixa_risco',
    'valor_renda', 'valor_emprestado', 'valor_taxa', 'valor_prazo',
    'classe_restritivo_cenario', 'classe_tempo_relacionamento',
    'multiplicador_renda_cenario', 'teto_rating_cenario',
    'pct_max_comprometimento_cenario',
    'limite_multiplicador_cenario', 'limite_capacidade_cenario',
    'limite_bruto_cenario', 'limite_final_cenario',
    'decisao_cenario', 'valor_aprovado_cenario',
    'target_inadimplente_12m',
]

resultados = {}
print('Simulando cenarios...')
print()

for nome, params in CENARIOS.items():
    df_res = simular_politica_limite(df, nome, params)
    resultados[nome] = df_res

    mask_aprov = df_res['decisao_cenario'].isin(['Aprovar valor solicitado', 'Aprovar valor reduzido'])
    aprov_pct  = mask_aprov.mean()
    manual_pct = (df_res['decisao_cenario'] == 'Análise manual').mean()
    recusa_pct = (df_res['decisao_cenario'] == 'Recusar').mean()
    br_aprov   = df_res.loc[mask_aprov, 'target_inadimplente_12m'].mean() if mask_aprov.any() else 0

    print(f'[{nome}] Aprov auto: {aprov_pct:.1%} | Manual: {manual_pct:.1%} | Recusa: {recusa_pct:.1%} | Bad rate aprov: {br_aprov:.2%}')
    validar_consistencia_politica(df_res, nome)
    print()

cols_presentes = [c for c in COLS_SAIDA if c in list(resultados.values())[0].columns]
df_cenarios = pd.concat(
    [v[cols_presentes] for v in resultados.values()],
    ignore_index=True
)
print(f'Base consolidada: {df_cenarios.shape[0]:,} linhas x {df_cenarios.shape[1]} colunas')

Simulando cenarios...

[Conservador] Aprov auto: 58.5% | Manual: 28.7% | Recusa: 12.9% | Bad rate aprov: 2.11%
  [Conservador] OK — consistencia validada



[Base/Equilibrado] Aprov auto: 60.1% | Manual: 29.7% | Recusa: 10.2% | Bad rate aprov: 2.23%
  [Base/Equilibrado] OK — consistencia validada



[Expansivo controlado] Aprov auto: 60.9% | Manual: 29.1% | Recusa: 10.0% | Bad rate aprov: 2.23%
  [Expansivo controlado] OK — consistencia validada

Base consolidada: 14,586 linhas x 21 colunas


In [16]:
ARQUIVO_SAIDA = PROCESSED_DIR / 'base_simulacao_cenarios_politica.parquet'
df_cenarios.to_parquet(ARQUIVO_SAIDA, index=False)
print(f'Base salva: {ARQUIVO_SAIDA}')

Base salva: C:\GitHub\datascience\projetos\concessao_credito\data\processed\base_simulacao_cenarios_politica.parquet


## 9. Impacto financeiro da política

In [17]:
def resumir_impacto_cenario(df_cenario):
    n         = len(df_cenario)
    mask_sol  = df_cenario['decisao_cenario'] == 'Aprovar valor solicitado'
    mask_red  = df_cenario['decisao_cenario'] == 'Aprovar valor reduzido'
    mask_man  = df_cenario['decisao_cenario'] == 'Análise manual'
    mask_rec  = df_cenario['decisao_cenario'] == 'Recusar'
    mask_aprov = mask_sol | mask_red
    df_aprov  = df_cenario[mask_aprov]
    val_sol   = df_cenario['valor_emprestado'].sum()
    val_apr   = df_cenario['valor_aprovado_cenario'].sum()
    return {
        'cenario':                         df_cenario['cenario'].iloc[0],
        'qtd_clientes':                    n,
        'taxa_historica_inadimplencia':    df_cenario['target_inadimplente_12m'].mean(),
        'taxa_aprovacao_valor_solicitado': mask_sol.mean(),
        'taxa_aprovacao_reduzida':         mask_red.mean(),
        'taxa_aprovacao_automatica_total': mask_aprov.mean(),
        'taxa_analise_manual':             mask_man.mean(),
        'taxa_recusa':                     mask_rec.mean(),
        'pd_media_aprovados':              df_aprov['pd_score'].mean() if len(df_aprov) > 0 else np.nan,
        'bad_rate_observado_aprovados':    df_aprov['target_inadimplente_12m'].mean() if len(df_aprov) > 0 else np.nan,
        'valor_solicitado_total':          val_sol,
        'valor_aprovado_total':            val_apr,
        'pct_exposicao_aprovada':          val_apr / val_sol if val_sol > 0 else 0,
        'reducao_exposicao':               val_sol - val_apr,
        'limite_medio':                    df_cenario['limite_final_cenario'].mean(),
        'valor_aprovado_medio':            df_aprov['valor_aprovado_cenario'].mean() if len(df_aprov) > 0 else 0,
    }


def resumir_impacto_por_rating(df_cenario):
    nome = df_cenario['cenario'].iloc[0]
    rows = []
    for rating in RATINGS_ORDER:
        grupo = df_cenario[df_cenario['faixa_risco'] == rating]
        if len(grupo) == 0:
            continue
        mask_aprov = grupo['decisao_cenario'].isin(['Aprovar valor solicitado', 'Aprovar valor reduzido'])
        df_aprov   = grupo[mask_aprov]
        val_sol    = grupo['valor_emprestado'].sum()
        val_apr    = grupo['valor_aprovado_cenario'].sum()
        rows.append({
            'cenario':                  nome,
            'faixa_risco':              rating,
            'qtd_clientes':             len(grupo),
            'pd_media':                 grupo['pd_score'].mean(),
            'bad_rate_observado':       grupo['target_inadimplente_12m'].mean(),
            'renda_media':              grupo['valor_renda'].mean(),
            'valor_solicitado_total':   val_sol,
            'valor_aprovado_total':     val_apr,
            'pct_exposicao_aprovada':   val_apr / val_sol if val_sol > 0 else 0,
            'limite_medio':             grupo['limite_final_cenario'].mean(),
            'valor_aprovado_medio':     df_aprov['valor_aprovado_cenario'].mean() if len(df_aprov) > 0 else 0,
            'taxa_aprovacao_automatica': mask_aprov.mean(),
            'taxa_analise_manual':      (grupo['decisao_cenario'] == 'Análise manual').mean(),
            'taxa_recusa':              (grupo['decisao_cenario'] == 'Recusar').mean(),
        })
    return pd.DataFrame(rows)


def resumir_impacto_por_decisao(df_cenario):
    nome    = df_cenario['cenario'].iloc[0]
    n_total = len(df_cenario)
    rows = []
    for decisao, grupo in df_cenario.groupby('decisao_cenario', observed=True):
        rows.append({
            'cenario':                nome,
            'decisao':                decisao,
            'qtd_clientes':           len(grupo),
            'participacao_clientes':  len(grupo) / n_total,
            'pd_media':               grupo['pd_score'].mean(),
            'bad_rate_observado':     grupo['target_inadimplente_12m'].mean(),
            'valor_solicitado_total': grupo['valor_emprestado'].sum(),
            'valor_aprovado_total':   grupo['valor_aprovado_cenario'].sum(),
            'valor_solicitado_medio': grupo['valor_emprestado'].mean(),
            'valor_aprovado_medio':   grupo['valor_aprovado_cenario'].mean(),
        })
    return pd.DataFrame(rows)


print('Funcoes de impacto definidas.')

Funcoes de impacto definidas.


In [18]:
df_impacto = pd.DataFrame([
    resumir_impacto_cenario(resultados[nome]) for nome in CENARIOS
])
df_impacto.to_csv(TABLES_DIR / 'politica_impacto_financeiro_cenarios.csv', index=False)

cols_show = [
    'cenario', 'taxa_aprovacao_automatica_total', 'taxa_analise_manual', 'taxa_recusa',
    'bad_rate_observado_aprovados', 'pct_exposicao_aprovada', 'valor_aprovado_total',
]
print('Impacto financeiro por cenario:')
print(df_impacto[cols_show].round(4).to_string(index=False))
print()
print('Salvo em: politica_impacto_financeiro_cenarios.csv')

Impacto financeiro por cenario:
             cenario  taxa_aprovacao_automatica_total  taxa_analise_manual  taxa_recusa  bad_rate_observado_aprovados  pct_exposicao_aprovada  valor_aprovado_total
         Conservador                           0.5845               0.2865       0.1290                        0.0211                  0.1982          1.280772e+07
    Base/Equilibrado                           0.6006               0.2972       0.1022                        0.0223                  0.2653          1.714272e+07
Expansivo controlado                           0.6086               0.2910       0.1004                        0.0223                  0.3226          2.084185e+07

Salvo em: politica_impacto_financeiro_cenarios.csv


In [19]:
df_impacto_rating = pd.concat(
    [resumir_impacto_por_rating(resultados[n]) for n in CENARIOS],
    ignore_index=True
)
df_impacto_rating.to_csv(TABLES_DIR / 'politica_impacto_por_rating.csv', index=False)

mask_base = df_impacto_rating['cenario'] == 'Base/Equilibrado'
cols_r = [
    'faixa_risco', 'qtd_clientes', 'pd_media', 'bad_rate_observado',
    'taxa_aprovacao_automatica', 'taxa_analise_manual', 'taxa_recusa',
    'pct_exposicao_aprovada', 'valor_aprovado_total',
]
print('Impacto por rating — Base/Equilibrado:')
print(df_impacto_rating[mask_base][cols_r].round(4).to_string(index=False))
print()
print('Salvo em: politica_impacto_por_rating.csv')

Impacto por rating — Base/Equilibrado:
          faixa_risco  qtd_clientes  pd_media  bad_rate_observado  taxa_aprovacao_automatica  taxa_analise_manual  taxa_recusa  pct_exposicao_aprovada  valor_aprovado_total
      A - Baixo risco          1945    0.0225              0.0108                     0.9995               0.0000       0.0005                  0.5607          1.193076e+07
B - Médio-baixo risco           972    0.0607              0.0453                     1.0000               0.0000       0.0000                  0.3652          5.192650e+06
      C - Médio risco           729    0.1098              0.1056                     0.0055               0.9822       0.0123                  0.0018          1.930916e+04
       D - Alto risco           729    0.2253              0.2510                     0.0000               1.0000       0.0000                  0.0000          0.000000e+00
 E - Muito alto risco           487    0.5224              0.5606                     0.0000    

In [20]:
df_impacto_decisao = pd.concat(
    [resumir_impacto_por_decisao(resultados[n]) for n in CENARIOS],
    ignore_index=True
)
df_impacto_decisao.to_csv(TABLES_DIR / 'politica_impacto_por_decisao.csv', index=False)

print('Impacto por decisao — todos os cenarios:')
print(df_impacto_decisao.round(4).to_string(index=False))
print()
print('Salvo em: politica_impacto_por_decisao.csv')

Impacto por decisao — todos os cenarios:
             cenario                  decisao  qtd_clientes  participacao_clientes  pd_media  bad_rate_observado  valor_solicitado_total  valor_aprovado_total  valor_solicitado_medio  valor_aprovado_medio
         Conservador           Análise manual          1393                 0.2865    0.1702              0.1845             21257341.34          0.000000e+00              15260.1158                0.0000
         Conservador   Aprovar valor reduzido          2443                 0.5025    0.0378              0.0229             33702870.34          1.129770e+07              13795.6899             4624.5209
         Conservador Aprovar valor solicitado           399                 0.0821    0.0165              0.0100              1510011.37          1.510011e+06               3784.4896             3784.4896
         Conservador                  Recusar           627                 0.1290    0.4233              0.4482              8134144.17   

## 10. Comparação dos cenários e recomendação

In [21]:
PALETA = [CORES['verde_principal'], CORES['verde_vivo'], CORES['amarelo'], CORES['marrom'], CORES['cinza_medio']]

# Grafico 1: Distribuicao por rating
fig1 = px.bar(
    tab_rating_pd, x='faixa_risco', y='qtd_clientes', color='faixa_risco',
    color_discrete_sequence=PALETA,
    labels={'faixa_risco': 'Rating', 'qtd_clientes': 'Qtd clientes'},
)
fig1 = aplicar_layout(fig1, titulo='Distribuicao de clientes por rating',
    subtitulo='Base de validacao com score de PD')
salvar_figura(fig1, caminho_html=str(FIGURES_DIR / '06_publico_distribuicao_rating.html'))
fig1.show()

# Grafico 2: Bad rate por rating
fig2 = px.bar(
    tab_rating_pd, x='faixa_risco', y='bad_rate_observado', color='faixa_risco',
    color_discrete_sequence=PALETA,
    labels={'faixa_risco': 'Rating', 'bad_rate_observado': 'Bad rate observado'},
)
fig2 = aplicar_layout(fig2, titulo='Bad rate observado por rating',
    subtitulo='Taxa historica de inadimplencia em 12 meses')
fig2.update_layout(yaxis_tickformat='.1%')
salvar_figura(fig2, caminho_html=str(FIGURES_DIR / '06_publico_bad_rate_rating.html'))
fig2.show()

# Grafico 3: Renda media por rating
fig3 = px.bar(
    tab_rating_exp, x='faixa_risco', y='renda_media', color='faixa_risco',
    color_discrete_sequence=PALETA,
    labels={'faixa_risco': 'Rating', 'renda_media': 'Renda media (R$)'},
)
fig3 = aplicar_layout(fig3, titulo='Renda media por rating', subtitulo='Base de validacao')
salvar_figura(fig3, caminho_html=str(FIGURES_DIR / '06_publico_renda_rating.html'))
fig3.show()

In [22]:
PALETA_CEN = [CORES['verde_principal'], CORES['amarelo'], CORES['marrom']]

# Grafico 4: Exposicao aprovada por cenario
fig4 = px.bar(
    df_impacto, x='cenario', y='valor_aprovado_total', color='cenario',
    color_discrete_sequence=PALETA_CEN,
    labels={'cenario': 'Cenario', 'valor_aprovado_total': 'Valor aprovado total (R$)'},
)
fig4 = aplicar_layout(fig4, titulo='Exposicao aprovada automaticamente por cenario',
    subtitulo='Soma do valor aprovado em cada cenario')
salvar_figura(fig4, caminho_html=str(FIGURES_DIR / '06_impacto_exposicao_cenarios.html'))
fig4.show()

# Grafico 5: Valor aprovado por rating — cenario Base
df_base_rating = df_impacto_rating[df_impacto_rating['cenario'] == 'Base/Equilibrado'].copy()
fig5 = px.bar(
    df_base_rating, x='faixa_risco', y='valor_aprovado_total', color='faixa_risco',
    color_discrete_sequence=PALETA,
    labels={'faixa_risco': 'Rating', 'valor_aprovado_total': 'Valor aprovado (R$)'},
)
fig5 = aplicar_layout(fig5, titulo='Valor aprovado total por rating — Base/Equilibrado',
    subtitulo='Exposicao gerada pela politica no cenario base')
salvar_figura(fig5, caminho_html=str(FIGURES_DIR / '06_impacto_por_rating.html'))
fig5.show()

# Grafico 6: Trade-off aprovacao vs inadimplencia
fig6 = px.scatter(
    df_impacto, x='taxa_aprovacao_automatica_total', y='bad_rate_observado_aprovados',
    text='cenario', color='cenario', color_discrete_sequence=PALETA_CEN,
    labels={
        'taxa_aprovacao_automatica_total': 'Taxa de aprovacao automatica',
        'bad_rate_observado_aprovados':    'Bad rate dos aprovados',
    },
)
fig6 = aplicar_layout(fig6, titulo='Trade-off: Aprovacao automatica x Bad rate dos aprovados',
    subtitulo='Cada ponto representa um cenario de apetite de risco')
fig6.update_layout(xaxis_tickformat='.1%', yaxis_tickformat='.1%')
fig6.update_traces(textposition='top center', marker_size=14)
salvar_figura(fig6, caminho_html=str(FIGURES_DIR / '06_tradeoff_aprovacao_inadimplencia.html'))
fig6.show()

# Grafico 7: Solicitado vs aprovado por cenario
df_vs = df_impacto[['cenario', 'valor_solicitado_total', 'valor_aprovado_total']].melt(
    id_vars='cenario', var_name='tipo', value_name='valor'
)
df_vs['tipo'] = df_vs['tipo'].map({
    'valor_solicitado_total': 'Solicitado',
    'valor_aprovado_total':   'Aprovado',
})
fig7 = px.bar(
    df_vs, x='cenario', y='valor', color='tipo', barmode='group',
    color_discrete_sequence=[CORES['verde_principal'], CORES['verde_vivo']],
    labels={'cenario': 'Cenario', 'valor': 'Valor (R$)', 'tipo': ''},
)
fig7 = aplicar_layout(fig7, titulo='Valor solicitado vs aprovado por cenario',
    subtitulo='Impacto da politica sobre a exposicao total')
salvar_figura(fig7, caminho_html=str(FIGURES_DIR / '06_valor_solicitado_vs_aprovado.html'))
fig7.show()

In [23]:
# Score gerencial de apoio a recomendacao
print('=== Score gerencial ? apoio a recomendacao ===')
print()
print('Pesos: inadimplencia=45%, aprovacao=20%, exposicao=15%, eficiencia operacional=20%')
print('Eficiencia operacional = aprovacao automatica total - taxa de analise manual.')
print()

df_score = df_impacto[[
    'cenario', 'taxa_aprovacao_automatica_total', 'taxa_analise_manual',
    'bad_rate_observado_aprovados', 'pct_exposicao_aprovada',
]].copy()

eps = 1e-9

def normalizar_maior_melhor(serie):
    minimo, maximo = serie.min(), serie.max()
    if abs(maximo - minimo) < eps:
        return pd.Series(1.0, index=serie.index)
    return (serie - minimo) / (maximo - minimo)


def normalizar_menor_melhor(serie):
    minimo, maximo = serie.min(), serie.max()
    if abs(maximo - minimo) < eps:
        return pd.Series(1.0, index=serie.index)
    return 1 - (serie - minimo) / (maximo - minimo)


df_score['taxa_eficiencia_operacional'] = (
    df_score['taxa_aprovacao_automatica_total'] - df_score['taxa_analise_manual']
)

df_score['score_inadimplencia'] = normalizar_menor_melhor(df_score['bad_rate_observado_aprovados'])
df_score['score_aprovacao']     = normalizar_maior_melhor(df_score['taxa_aprovacao_automatica_total'])
df_score['score_exposicao']     = normalizar_maior_melhor(df_score['pct_exposicao_aprovada'])
df_score['score_eficiencia']    = normalizar_maior_melhor(df_score['taxa_eficiencia_operacional'])

df_score['score_gerencial'] = (
    df_score['score_inadimplencia'] * 0.45 +
    df_score['score_aprovacao']     * 0.20 +
    df_score['score_exposicao']     * 0.15 +
    df_score['score_eficiencia']    * 0.20
)
df_score = df_score.sort_values('score_gerencial', ascending=False).reset_index(drop=True)
df_score.to_csv(TABLES_DIR / 'politica_score_gerencial_cenarios.csv', index=False)

cols_s = [
    'cenario', 'score_inadimplencia', 'score_aprovacao', 'score_exposicao',
    'score_eficiencia', 'score_gerencial',
]
print(df_score[cols_s].round(4).to_string(index=False))
print()

CENARIO_RECOMENDADO = df_score.iloc[0]['cenario']
print(f'Score gerencial sugere: {CENARIO_RECOMENDADO}')
print()
print('Nota: o score gerencial e apoio de decisao, nao verdade absoluta.')
print('O peso de exposicao foi limitado para nao escolher automaticamente o cenario expansivo.')
print('A escolha final deve ser validada com a area de politica de credito.')
print()
print('Salvo em: politica_score_gerencial_cenarios.csv')

=== Score gerencial ? apoio a recomendacao ===

Pesos: inadimplencia=45%, aprovacao=20%, exposicao=15%, eficiencia operacional=20%
Eficiencia operacional = aprovacao automatica total - taxa de analise manual.

             cenario  score_inadimplencia  score_aprovacao  score_exposicao  score_eficiencia  score_gerencial
Expansivo controlado               0.0000           1.0000           1.0000            1.0000           0.5500
         Conservador               1.0000           0.0000           0.0000            0.0000           0.4500
    Base/Equilibrado               0.0374           0.6667           0.5396            0.2737           0.2858

Score gerencial sugere: Expansivo controlado

Nota: o score gerencial e apoio de decisao, nao verdade absoluta.
O peso de exposicao foi limitado para nao escolher automaticamente o cenario expansivo.
A escolha final deve ser validada com a area de politica de credito.

Salvo em: politica_score_gerencial_cenarios.csv


### Justificativa da recomendação

O score gerencial pondera quatro dimensões:

- **Controle de inadimplência (45%)**: o cenário que aprova clientes com menor bad rate histórico recebe maior peso, pois proteger a qualidade da carteira é o objetivo central.
- **Aprovação automática (20%)**: maior aprovação reduz fricção operacional, desde que preserve controles de risco.
- **Exposição aprovada (15%)**: maior exposição representa maior receita potencial, mas tem peso limitado para não favorecer automaticamente o cenário expansivo.
- **Eficiência operacional (20%)**: quando a taxa de análise manual não diferencia os cenários, a eficiência é medida por aprovação automática líquida de análise manual.

O cenário recomendado é definido pelo score como apoio de decisão e deve ser interpretado junto com as métricas de risco. O **Expansivo controlado** só é defensável se mantiver bad rate de aprovados próxima ao cenário base e preservar os controles adicionados para rating C; caso o apetite de risco seja mais restrito, o **Base/Equilibrado** continua sendo a alternativa prudencial.

> A escolha final deve considerar o apetite de risco da instituição, metas de carteira e validação com a área de política de crédito.

## 11. Política final recomendada

In [24]:
print(f'Politica final recomendada: {CENARIO_RECOMENDADO}')
print()

params_rec    = CENARIOS[CENARIO_RECOMENDADO]
df_rec_rating = df_impacto_rating[df_impacto_rating['cenario'] == CENARIO_RECOMENDADO]

rows_final = []
for rating in RATINGS_ORDER:
    row_r      = df_rec_rating[df_rec_rating['faixa_risco'] == rating]
    row_pd     = tab_rating_pd[tab_rating_pd['faixa_risco'] == rating]
    bad_rate   = float(row_r['bad_rate_observado'].values[0]) if len(row_r) > 0 else float('nan')
    pd_min     = float(row_pd['pd_min'].values[0])            if len(row_pd) > 0 else float('nan')
    pd_max     = float(row_pd['pd_max'].values[0])            if len(row_pd) > 0 else float('nan')
    intervalo_pd = f'{pd_min:.2%} a {pd_max:.2%}'.replace('.', ',')
    tratamento = params_rec['tratamento_rating'].get(rating, 'recusa')
    mult       = params_rec['multiplicadores'].get(rating, 0.0)
    teto       = params_rec['teto_rating'].get(rating, 0.0)
    pct_comp   = params_rec['pct_max_comprometimento'].get(rating, 0.0)

    if tratamento == 'automatico' and rating == 'C - Médio risco':
        acao = 'Aprovar valor solicitado / Aprovar valor reduzido; Análise manual em casos de controle adicional'
    else:
        acao = {
            'automatico':     'Aprovar valor solicitado / Aprovar valor reduzido',
            'analise_manual': 'Análise manual',
            'recusa':         'Recusar',
        }.get(tratamento, 'Recusar')

    if tratamento == 'automatico':
        regra = 'limite = min(renda x {:.1f}, VP_parcela_max, R$ {:,.0f})'.format(mult, teto)
    else:
        regra = 'Sem limite automatico'

    rows_final.append({
        'cenario_recomendado':             CENARIO_RECOMENDADO,
        'faixa_risco':                     rating,
        'intervalo_pd':                    intervalo_pd,
        'bad_rate_observado':              bad_rate,
        'multiplicador_renda':             mult,
        'teto_rating':                     teto,
        'pct_max_comprometimento':         pct_comp,
        'tratamento_restritivos':          str(params_rec['fatores_restritivo']),
        'tratamento_cliente_inativo':      'fator = {:.2f}'.format(params_rec['fatores_cliente_ativo'].get(0, 1.0)),
        'tratamento_tempo_relacionamento': str(params_rec['fatores_tempo_relacionamento']),
        'acao_principal':                  acao,
        'regra_limite':                    regra,
        'observacao_negocio': (
            'Limite = min(renda x mult, VP parcela max, teto) x fator_rest x fator_ativo x fator_tempo. '
            'Rating C exige mesa para restritivo relevante, cliente inativo, relacionamento curto ou proposta muito acima do limite. '
            'Parametros a validar com politica de credito antes de qualquer uso produtivo.'
        ),
    })

df_politica_final = pd.DataFrame(rows_final)
df_politica_final.to_csv(TABLES_DIR / 'politica_final_recomendada_limites.csv', index=False)

cols_exib = [
    'faixa_risco', 'bad_rate_observado', 'multiplicador_renda',
    'teto_rating', 'pct_max_comprometimento', 'acao_principal',
]
print(df_politica_final[cols_exib].round(4).to_string(index=False))
print()
print('Salvo em: politica_final_recomendada_limites.csv')

Politica final recomendada: Expansivo controlado

          faixa_risco  bad_rate_observado  multiplicador_renda  teto_rating  pct_max_comprometimento                                                                                   acao_principal
      A - Baixo risco              0.0108                  5.0     39222.68                     0.35                                                Aprovar valor solicitado / Aprovar valor reduzido
B - Médio-baixo risco              0.0453                  4.0     31378.15                     0.30                                                Aprovar valor solicitado / Aprovar valor reduzido
      C - Médio risco              0.1056                  2.5     23533.61                     0.25 Aprovar valor solicitado / Aprovar valor reduzido; Análise manual em casos de controle adicional
       D - Alto risco              0.2510                  0.0         0.00                     0.20                                                          

## 12. Limitações, governança e próximos passos

### 12.1 Limitações metodológicas

- **Base apenas com operações concedidas**: não há propostas recusadas. O perfil de risco simulado reflete clientes aprovados no passado (viés de seleção).
- **Ausência de garantia ou colateral**: nenhuma garantia real ou pessoal foi modelada.
- **Ausência de LGD**: todos os defaults são tratados como equivalentes. A perda dado o default não está considerada.
- **Ausência de EAD formal**: o `valor_emprestado` histórico é usado como proxy do valor exposto.
- **Ausência de score externo completo**: a política usa apenas score interno derivado de variáveis da própria base.
- **Ausência de histórico detalhado de atraso**: não há curvas de atraso nem recuperação de crédito.
- **Simulação histórica ≠ política de produção**: bad rates observados são históricos e não garantem comportamento futuro.
- **Parâmetros iniciais**: multiplicadores, tetos e fatores foram calibrados como ponto de partida. Precisam de validação com a área de política de crédito antes de qualquer implantação.
- **Variáveis sensíveis**: `idade_concessao` e `cat_escolaridade` exigem avaliação de governança e fairness.
- **Ausência de política para PJ**: esta política trata apenas pessoa física com relacionamento bancário.

### 12.2 Monitoramento recomendado

Após implantação, monitorar mensalmente:

- Taxa de aprovação automática (solicitado e reduzido)
- Taxa de análise manual e recusa
- Valor aprovado e exposição por rating
- Bad rate por safra e por rating
- PD média dos aprovados
- AUC e KS do score (estabilidade do modelo)
- PSI do `pd_score` e das variáveis principais
- Concentração de decisão por grupo de risco
- Perda observada, quando LGD/EAD forem disponibilizados

### 12.3 Próximos passos

1. Validar parâmetros com a área de política de crédito
2. Definir governança para variáveis sensíveis (idade, escolaridade)
3. Implementar esteira de monitoramento por safra
4. Incorporar dados de propostas recusadas quando disponíveis (reduz viés de seleção)
5. Avaliar inclusão de LGD/EAD quando houver base suficiente
6. Revisar multiplicadores e tetos após 6 meses de operação

## 13. Checklist de saídas

In [25]:
print('=== Checklist de saidas do notebook 06 ===')
print()

saidas = [
    ('Parquet simulacao',           PROCESSED_DIR / 'base_simulacao_cenarios_politica.parquet'),
    ('Publico: renda x rating',     TABLES_DIR / 'politica_publico_faixa_renda_rating.csv'),
    ('Publico: rating x tempo',     TABLES_DIR / 'politica_publico_rating_tempo_relacionamento.csv'),
    ('Publico: rating x PD',        TABLES_DIR / 'politica_publico_rating_pd_bad_rate.csv'),
    ('Publico: rating x exposicao', TABLES_DIR / 'politica_publico_rating_exposicao.csv'),
    ('Publico: rating x restrict.', TABLES_DIR / 'politica_publico_rating_restritivos.csv'),
    ('Parametros limite',           TABLES_DIR / 'politica_parametros_limite_cenarios.csv'),
    ('Impacto financeiro',          TABLES_DIR / 'politica_impacto_financeiro_cenarios.csv'),
    ('Impacto por rating',          TABLES_DIR / 'politica_impacto_por_rating.csv'),
    ('Impacto por decisao',         TABLES_DIR / 'politica_impacto_por_decisao.csv'),
    ('Score gerencial',             TABLES_DIR / 'politica_score_gerencial_cenarios.csv'),
    ('Politica final',              TABLES_DIR / 'politica_final_recomendada_limites.csv'),
    ('Graf: dist rating',           FIGURES_DIR / '06_publico_distribuicao_rating.html'),
    ('Graf: bad rate rating',       FIGURES_DIR / '06_publico_bad_rate_rating.html'),
    ('Graf: renda rating',          FIGURES_DIR / '06_publico_renda_rating.html'),
    ('Graf: exposicao cenarios',    FIGURES_DIR / '06_impacto_exposicao_cenarios.html'),
    ('Graf: impacto por rating',    FIGURES_DIR / '06_impacto_por_rating.html'),
    ('Graf: tradeoff',              FIGURES_DIR / '06_tradeoff_aprovacao_inadimplencia.html'),
    ('Graf: sol vs aprov',          FIGURES_DIR / '06_valor_solicitado_vs_aprovado.html'),
]

todos_ok = True
for descricao, path in saidas:
    existe = path.exists()
    status = 'OK' if existe else 'AUSENTE'
    if not existe:
        todos_ok = False
    print(f'  [{status}] {descricao:30s} -> {path.name}')

print()
print('Status final:', 'TODAS AS SAIDAS GERADAS.' if todos_ok else 'ATENCAO — alguma saida ausente.')
print()
print(f'Cenario recomendado: {CENARIO_RECOMENDADO}')
print()
print('A politica recomendada e uma politica inicial de limite baseada em evidencia historica.')
print('O score de PD foi usado como rating interno. A decisao combina rating, renda,')
print('teto de limite, capacidade de pagamento, restritivos e relacionamento.')
print('A recomendacao deve ser validada com a area de politica de credito antes de uso produtivo.')

=== Checklist de saidas do notebook 06 ===

  [OK] Parquet simulacao              -> base_simulacao_cenarios_politica.parquet
  [OK] Publico: renda x rating        -> politica_publico_faixa_renda_rating.csv
  [OK] Publico: rating x tempo        -> politica_publico_rating_tempo_relacionamento.csv
  [OK] Publico: rating x PD           -> politica_publico_rating_pd_bad_rate.csv
  [OK] Publico: rating x exposicao    -> politica_publico_rating_exposicao.csv
  [OK] Publico: rating x restrict.    -> politica_publico_rating_restritivos.csv
  [OK] Parametros limite              -> politica_parametros_limite_cenarios.csv
  [OK] Impacto financeiro             -> politica_impacto_financeiro_cenarios.csv
  [OK] Impacto por rating             -> politica_impacto_por_rating.csv
  [OK] Impacto por decisao            -> politica_impacto_por_decisao.csv
  [OK] Score gerencial                -> politica_score_gerencial_cenarios.csv
  [OK] Politica final                 -> politica_final_recomendada_limit